#### Loading de dados

In [1]:
import numpy as np
import pandas as pd
from lightweight_charts import JupyterChart

df = pd.read_parquet('/Users/marcelogarcia/Documents/Development/autobt/autobt/data/usatec2017_2025.parquet')

#### Funções de sinais

In [2]:
"""
================================================================================
CANDLE STRENGTH INDICATOR v3 — Hybrid
================================================================================

Base: Estrutura multiplicativa + ATR sizing + hard volume gate (elephant bar)
Adições: Shadow boost convexo + doji handling + gate ergódico + concentração

MUDANÇAS vs ELEPHANT BAR ORIGINAL:
1. shape agora é (body_share + shadow_weight * rejection^α) * close_extreme * (1 - opp_wick)
   → Sombras favoráveis CONTRIBUEM mesmo com corpo pequeno (hammers, dojis)
2. Dojis: direction via close vs midpoint (dragonfly/gravestone não são zerados)
3. Gate ergódico multiplicativo (pós-numba): atenua candles típicos para seu horário
4. Concentração calibrável: P(|CSI|>0.9) ≈ 2% (configurável)

TUDO QUE NÃO ESTÁ LISTADO ACIMA ESTÁ IDÊNTICO AO ORIGINAL.
================================================================================
"""

import numpy as np
from typing import Optional, Dict, Tuple
from datetime import datetime, timezone, timedelta

# Numba fallback: usa njit se disponível, senão no-op decorator
try:
    from numba import njit
except ImportError:
    def njit(*args, **kwargs):
        """Fallback: roda como Python puro se numba não disponível."""
        def decorator(func):
            return func
        if len(args) == 1 and callable(args[0]):
            return args[0]
        return decorator


# ==============================================================================
# HELPERS (idênticos ao original)
# ==============================================================================

@njit(cache=True, nogil=True)
def _clip01(x):
    if x < 0.0:
        return 0.0
    if x > 1.0:
        return 1.0
    return x


@njit(cache=True, nogil=True)
def _ema_fast(arr, period):
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    if n == 0:
        return out
    p = period
    if p <= 1:
        p = 1
    alpha = 2.0 / (p + 1.0)
    one_minus_alpha = 1.0 - alpha
    out[0] = arr[0]
    for i in range(1, n):
        out[i] = alpha * arr[i] + one_minus_alpha * out[i - 1]
    return out


# ==============================================================================
# CORE NUMBA — ELEPHANT BAR ENHANCED
# ==============================================================================

@njit(cache=True, nogil=True)
def _candle_strength_core(
    open_,
    close_,
    high_,
    low_,
    volume_,
    window_size=1000,
    short_window=20,
    volume_rel_min=1.0,
    # --- NOVOS PARAMS ---
    shadow_alpha=2.0,     # expoente convexo para sombras favoráveis
    shadow_weight=0.40,   # peso do boost de sombra no shape
):
    """
    Core multiplicativo em Numba.
    
    MUDANÇAS vs original:
    
    1) SHAPE com shadow boost:
       ANTES:  shape = body_share * close_extreme * (1 - opp_wick)
       DEPOIS: shape = (body_share + sw * rejection^α) * close_extreme * (1 - opp_wick)
       
       Onde:
         rejection = favorable_shadow_ratio ^ shadow_alpha
         favorable = lower_wick/range (bullish) ou upper_wick/range (bearish)
       
       Por que: um hammer com 88% de sombra inferior e corpo de 8% tinha
       shape ≈ 0.06 no original. Com shadow boost (α=2, sw=0.4):
       shape ≈ 0.06 + 0.4*(0.88²)*0.92*0.95 ≈ 0.33 → captura a rejeição.
    
    2) DOJI handling:
       ANTES:  close == open → direction = 0 → strength = 0 (SEMPRE)
       DEPOIS: close == open → direction via close vs midpoint do range
       
       Dragonfly doji (close=high, long lower shadow):
         direction = +1 (close > midpoint)
         body_share = 0, MAS shadow_boost contribui
         → shape ≈ 0.4 * 1.0 * 1.0 * 1.0 = 0.40 (sinal não-trivial)
       
       Doji simétrico (close ≈ midpoint):
         direction = 0 → strength = 0 (comportamento original preservado)
    """
    n = close_.shape[0]
    strength = np.empty(n, dtype=np.float64)
    shape_out = np.empty(n, dtype=np.float64)
    size_out = np.empty(n, dtype=np.float64)
    volume_gate_out = np.empty(n, dtype=np.float64)
    volume_rel_out = np.empty(n, dtype=np.float64)

    if n == 0:
        return strength, shape_out, size_out, volume_gate_out, volume_rel_out

    long_window = window_size
    if long_window < 20:
        long_window = 20

    short_w = short_window
    if short_w < 2:
        short_w = 2

    volume_window = long_window // 5
    if volume_window < 20:
        volume_window = 20

    # --- True Range + ATR + Volume EMA (idêntico ao original) ---
    tr = np.empty(n, dtype=np.float64)
    tr[0] = high_[0] - low_[0]
    for i in range(1, n):
        a = high_[i] - low_[i]
        b = abs(high_[i] - close_[i - 1])
        c = abs(low_[i] - close_[i - 1])
        m = a
        if b > m:
            m = b
        if c > m:
            m = c
        tr[i] = m

    atr_short = _ema_fast(tr, short_w)
    atr_long = _ema_fast(tr, long_window)
    volume_ema = _ema_fast(volume_, volume_window)

    eps = 1e-12

    for i in range(n):
        o = open_[i]
        c = close_[i]
        h = high_[i]
        l = low_[i]

        body = abs(c - o)
        r = h - l
        if r < eps:
            r = eps

        # --- MUDANÇA 1: Direction com doji handling ---
        if c > o:
            direction = 1.0
        elif c < o:
            direction = -1.0
        else:
            # Doji: usar posição do close vs midpoint do range
            mid = (h + l) * 0.5
            if c > mid + eps:
                direction = 1.0
            elif c < mid - eps:
                direction = -1.0
            else:
                direction = 0.0

        upper = h - max(c, o)
        lower = min(c, o) - l

        body_share = _clip01(body / r)

        if direction > 0.0:
            close_extreme = _clip01((c - l) / r)
            opposite_wick = _clip01(upper / r)
            favorable_wick = _clip01(lower / r)
        elif direction < 0.0:
            close_extreme = _clip01((h - c) / r)
            opposite_wick = _clip01(lower / r)
            favorable_wick = _clip01(upper / r)
        else:
            close_extreme = 0.0
            opposite_wick = 1.0
            favorable_wick = 0.0

        # --- MUDANÇA 2: Shape com shadow boost ---
        # Transformação convexa: sombras grandes contribuem desproporcionalmente
        shadow_rejection = favorable_wick ** shadow_alpha
        shadow_boost = shadow_weight * shadow_rejection

        # Estrutura multiplicativa preservada, mas com termo aditivo no body_share
        # Quando body_share é 0 (doji), shadow_boost ainda contribui
        effective_body = body_share + shadow_boost
        if effective_body > 1.0:
            effective_body = 1.0

        shape = effective_body * close_extreme * (1.0 - opposite_wick)

        # --- SIZE (idêntico ao original) ---
        atr_s = atr_short[i]
        if atr_s < eps:
            atr_s = eps
        atr_l = atr_long[i]
        if atr_l < eps:
            atr_l = eps

        impulse_short = body / atr_s
        impulse_long = body / atr_l
        range_impulse_long = r / atr_l

        body_power = np.sqrt(max(impulse_short, 0.0) * max(impulse_long, 0.0))
        range_power = np.sqrt(max(range_impulse_long, 0.0) * max(impulse_long, 0.0))

        # MUDANÇA MÍNIMA: para dojis, usar range como proxy de impulso
        # Damped por 0.35 para não inflar dojis medianos
        if body < eps and direction != 0.0:
            doji_impulse_s = r / atr_s
            doji_impulse_l = r / atr_l
            body_power = np.sqrt(max(doji_impulse_s, 0.0) * max(doji_impulse_l, 0.0)) * 0.35

        size_raw = 0.95 * body_power + 0.05 * (range_power * body_share)
        size = np.tanh(max(size_raw, 0.0) * 0.55)
        size = _clip01(size)

        # --- VOLUME GATE (idêntico ao original) ---
        vol_base = volume_ema[i]
        if vol_base < eps:
            vol_base = eps
        vol_rel = volume_[i] / vol_base

        if vol_rel < 0.7:
            volume_gate = 0.0
        else:
            z = (vol_rel - volume_rel_min) / 0.12
            logistic = 1.0 / (1.0 + np.exp(-z))
            volume_gate = 0.20 + 0.80 * logistic

        # --- FINAL (idêntico ao original) ---
        core_strength = 0.68 * size + 0.32 * shape
        magnitude = core_strength * volume_gate
        if magnitude > 1.0:
            magnitude = 1.0

        strength[i] = direction * magnitude
        shape_out[i] = shape
        size_out[i] = size
        volume_gate_out[i] = volume_gate
        volume_rel_out[i] = vol_rel

    return strength, shape_out, size_out, volume_gate_out, volume_rel_out


# ==============================================================================
# GATE ERGÓDICO (numpy — pós-numba)
# ==============================================================================

def _parse_timestamps(timestamps):
    """Converte qualquer formato para datetime objects."""
    if len(timestamps) == 0:
        return np.array([], dtype=object)
    s = timestamps[0]
    if isinstance(s, datetime):
        return np.asarray(timestamps, dtype=object)
    if isinstance(s, np.datetime64):
        return np.array([
            datetime.fromtimestamp(
                (t - np.datetime64('1970-01-01T00:00:00')) / np.timedelta64(1, 's'),
                tz=timezone.utc)
            for t in timestamps], dtype=object)
    if isinstance(s, (int, float, np.integer, np.floating)):
        return np.array([datetime.fromtimestamp(float(t), tz=timezone.utc)
                         for t in timestamps], dtype=object)
    if isinstance(s, str):
        return np.array([datetime.fromisoformat(t.replace('Z', '+00:00'))
                         for t in timestamps], dtype=object)
    raise ValueError(f"Tipo de timestamp não reconhecido: {type(s)}")


def _detect_timeframe(timestamps):
    """Auto-detecta timeframe pela mediana dos deltas."""
    dt = _parse_timestamps(timestamps)
    if len(dt) < 2:
        return {'median_seconds': 86400, 'label': 'daily', 'category': 'daily'}
    deltas = [(dt[i] - dt[i-1]).total_seconds()
              for i in range(1, len(dt))
              if (dt[i] - dt[i-1]).total_seconds() > 0]
    if not deltas:
        return {'median_seconds': 86400, 'label': 'daily', 'category': 'daily'}
    md = np.median(deltas)
    if md < 60:     return {'median_seconds': md, 'label': f'{int(md)}s', 'category': 'sub_minute'}
    elif md < 300:  return {'median_seconds': md, 'label': f'{int(md/60)}min', 'category': 'minute'}
    elif md < 900:  return {'median_seconds': md, 'label': f'{int(md/60)}min', 'category': 'multi_minute'}
    elif md < 3600: return {'median_seconds': md, 'label': f'{int(md/60)}min', 'category': 'sub_hour'}
    elif md < 14400:return {'median_seconds': md, 'label': f'{int(md/3600)}H', 'category': 'hourly'}
    elif md < 604800:return {'median_seconds': md, 'label': 'daily', 'category': 'daily'}
    elif md < 2592000:return {'median_seconds': md, 'label': 'weekly', 'category': 'weekly'}
    else:           return {'median_seconds': md, 'label': 'monthly', 'category': 'monthly'}


def _create_temporal_buckets(timestamps, tf_info):
    """Cria buckets fine + coarse para ergodic gate."""
    dt = _parse_timestamps(timestamps)
    n = len(dt)
    cat = tf_info['category']
    fine = np.zeros(n, dtype=np.int32)
    coarse = np.zeros(n, dtype=np.int32)
    for i, d in enumerate(dt):
        h, m, dow = d.hour, d.minute, d.weekday()
        if cat == 'sub_minute':
            fine[i], coarse[i] = h*60+m, h
        elif cat == 'minute':
            fine[i], coarse[i] = (h*60+m)//5, h
        elif cat == 'multi_minute':
            s15 = (h*60+m)//15
            fine[i], coarse[i] = s15*7+dow, s15
        elif cat == 'sub_hour':
            fine[i], coarse[i] = h*7+dow, h
        elif cat == 'hourly':
            sess = 0 if h < 8 else (1 if h < 14 else 2)
            fine[i], coarse[i] = sess*7+dow, sess
        elif cat == 'daily':
            fine[i] = coarse[i] = dow
        elif cat == 'weekly':
            fine[i] = coarse[i] = min((d.day-1)//7, 4)
        else:
            fine[i] = coarse[i] = d.month - 1
    return fine, coarse


def _compute_ergodic_gate(
    values,
    fine_buckets,
    coarse_buckets,
    min_samples_fine=20,
    min_samples_coarse=10,
    lambda_max=0.35,
    eta=0.8,
):
    """
    Gate ergódico multiplicativo.
    
    unusualness = |2 * rank - 1|   (0 = típico, 1 = excepcional)
    gate = 1 - λ * conf * (1 - unusualness^η)
    
    Gate ∈ (1-λ, 1.0] — NUNCA infla, apenas atenua ou confirma.
    Causalidade estrita: só usa dados passados.
    """
    n = len(values)
    ergodic_ranks = np.full(n, 0.5)
    confidence = np.zeros(n)
    gate = np.ones(n)

    fine_hist = {}
    coarse_hist = {}

    for i in range(n):
        fb, cb = int(fine_buckets[i]), int(coarse_buckets[i])
        val = values[i]
        rank, conf = 0.5, 0.0

        if fb in fine_hist and len(fine_hist[fb]) >= min_samples_fine:
            past = np.array(fine_hist[fb])
            rank = np.sum(past <= val) / len(past)
            conf = 1.0
        elif cb in coarse_hist and len(coarse_hist[cb]) >= min_samples_coarse:
            past = np.array(coarse_hist[cb])
            rank = np.sum(past <= val) / len(past)
            conf = 0.7

        ergodic_ranks[i] = rank
        confidence[i] = conf

        unusualness = abs(2.0 * rank - 1.0)
        gate[i] = 1.0 - lambda_max * conf * (1.0 - unusualness ** eta)

        fine_hist.setdefault(fb, []).append(val)
        coarse_hist.setdefault(cb, []).append(val)

    return gate, ergodic_ranks, confidence


# ==============================================================================
# CONCENTRAÇÃO CALIBRADA
# ==============================================================================

def _concentration_exponent(threshold=0.9, tail_pct=0.02):
    """
    p = ln(threshold) / ln(1 - tail_pct)
    
    threshold=0.9, tail_pct=0.02 → p=5.22 → ~2% com |CSI|>0.9
    """
    return np.log(threshold) / np.log(1.0 - tail_pct)


def _concentrate(values, p):
    """sign(x) * |x|^p — comprime distribuição ao redor de zero."""
    return np.sign(values) * np.power(np.abs(np.clip(values, -1, 1)), p)


# ==============================================================================
# FUNÇÃO PRINCIPAL
# ==============================================================================

def candle_strength_indicator(
    open_: np.ndarray,
    close_: np.ndarray,
    high_: np.ndarray,
    low_: np.ndarray,
    volume_: np.ndarray,
    timestamps: Optional[np.ndarray] = None,
    # --- Params do core (elephant bar original) ---
    window_size: int = 1000,
    short_window: int = 20,
    volume_rel_min: float = 1.0,
    # --- Params novos (shadow boost) ---
    shadow_alpha: float = 2.0,
    shadow_weight: float = 0.40,
    # --- Gate ergódico ---
    ergodic_lambda: float = 0.35,
    ergodic_eta: float = 0.8,
    min_samples_fine: int = 20,
    min_samples_coarse: int = 10,
    # --- Concentração ---
    concentration_threshold: float = 0.9,
    concentration_tail_pct: float = 0.02,
    use_concentration: bool = False,
    # --- Output ---
    return_components: bool = False,
) -> Dict[str, np.ndarray]:
    """
    Candle Strength Indicator v3 — Hybrid.
    
    Pipeline:
    1. Core Numba (elephant bar enhanced) → strength bruto ∈ [-1, +1]
    2. Concentração (opcional) → comprime distribuição
    3. Gate ergódico (se timestamps) → atenua candles típicos p/ seu horário
    
    Params:
    -------
    open_, close_, high_, low_, volume_: arrays OHLCV
    timestamps: array de timestamps (qualquer formato). None = sem ergódico.
    window_size: janela longa para ATR (default 1000)
    short_window: janela curta para ATR (default 20)
    volume_rel_min: threshold de volume relativo para logistic gate
    shadow_alpha: expoente convexo para sombras favoráveis (default 2.0)
    shadow_weight: peso do shadow boost no shape (default 0.40)
    ergodic_lambda: atenuação máxima do gate ergódico (default 0.35)
    ergodic_eta: generosidade do gate (default 0.8)
    min_samples_fine/coarse: mínimos para buckets ergódicos
    concentration_threshold: limiar para calibração (default 0.9)
    concentration_tail_pct: fração acima do threshold (default 0.02 = 2%)
    use_concentration: True/False para ativar concentração
    return_components: se True, retorna intermediários
    
    Retorna:
    --------
    dict com 'csi' ∈ [-1, +1] e opcionalmente componentes.
    """
    # Garantir contiguidade (Numba exige)
    o = np.ascontiguousarray(open_, dtype=np.float64)
    c = np.ascontiguousarray(close_, dtype=np.float64)
    h = np.ascontiguousarray(high_, dtype=np.float64)
    l = np.ascontiguousarray(low_, dtype=np.float64)
    v = np.ascontiguousarray(volume_, dtype=np.float64)

    # === Estágio 1: Core Numba ===
    strength, shape, size, vol_gate, vol_rel = _candle_strength_core(
        o, c, h, l, v,
        window_size=window_size,
        short_window=short_window,
        volume_rel_min=volume_rel_min,
        shadow_alpha=shadow_alpha,
        shadow_weight=shadow_weight,
    )

    csi = strength.copy()

    # === Estágio 2: Concentração (opcional) ===
    conc_p = None
    if use_concentration:
        conc_p = _concentration_exponent(concentration_threshold,
                                          concentration_tail_pct)
        csi = _concentrate(csi, conc_p)

    # === Estágio 3: Gate ergódico (se timestamps fornecidos) ===
    has_ergodic = timestamps is not None and len(timestamps) > 0
    ergodic_gate = np.ones(len(csi))
    ergodic_rank = np.full(len(csi), 0.5)
    ergodic_conf = np.zeros(len(csi))
    tf_info = None

    if has_ergodic:
        tf_info = _detect_timeframe(timestamps)
        fine_b, coarse_b = _create_temporal_buckets(timestamps, tf_info)
        ergodic_gate, ergodic_rank, ergodic_conf = _compute_ergodic_gate(
            np.abs(strength),  # Gate opera sobre magnitude absoluta
            fine_b, coarse_b,
            min_samples_fine=min_samples_fine,
            min_samples_coarse=min_samples_coarse,
            lambda_max=ergodic_lambda,
            eta=ergodic_eta,
        )
        csi = csi * ergodic_gate

    csi = np.clip(csi, -1.0, 1.0)

    result = {'csi': csi}

    if return_components:
        result.update({
            'strength_raw': strength,
            'shape': shape,
            'size': size,
            'volume_gate': vol_gate,
            'volume_rel': vol_rel,
            'ergodic_gate': ergodic_gate,
            'ergodic_rank': ergodic_rank,
            'ergodic_confidence': ergodic_conf,
            'concentration_p': conc_p,
            'timeframe_info': tf_info,
        })

    return result


# ==============================================================================
# EXEMPLOS
# ==============================================================================

def run_examples():
    print("=" * 76)
    print("CSI v3 HYBRID — VALIDAÇÃO")
    print("=" * 76)

    # =================================================================
    # PARTE 1: SHADOW BOOST — IMPACTO EM CANDLES CLÁSSICOS
    # =================================================================
    print("\n" + "-" * 76)
    print("PARTE 1: SHADOW BOOST — ORIGINAL vs ENHANCED")
    print("-" * 76 + "\n")

    # Simular uma série mínima para ATR warm-up + candle de teste
    np.random.seed(42)
    n_warmup = 100
    base_price = 100.0

    # Warmup: candles normais com range ~1.0
    wo = np.full(n_warmup, base_price)
    wc = wo + np.random.normal(0, 0.3, n_warmup)
    wh = np.maximum(wo, wc) + np.abs(np.random.normal(0, 0.2, n_warmup))
    wl = np.minimum(wo, wc) - np.abs(np.random.normal(0, 0.2, n_warmup))
    wv = np.full(n_warmup, 1000.0)

    test_candles = [
        ("Marubozu Bullish",  100.0, 110.0, 110.0, 100.0, 1500.0),
        ("Marubozu Bearish",  110.0, 100.0, 110.0, 100.0, 1500.0),
        ("Hammer clássico",   108.0, 108.5, 109.0, 100.0, 1500.0),
        ("Shooting Star",     102.0, 101.5, 110.0, 101.0, 1500.0),
        ("Dragonfly Doji",    110.0, 110.0, 110.0, 100.0, 1500.0),
        ("Gravestone Doji",   100.0, 100.0, 110.0, 100.0, 1500.0),
        ("Doji simétrico",    105.0, 105.0, 108.0, 102.0, 1500.0),
        ("Hammer vol baixo",  108.0, 108.5, 109.0, 100.0, 300.0),
    ]

    print(f"  {'Candle':<22} {'Original':>10} {'Enhanced':>10} {'Δ':>8}  Notas")
    print("  " + "-" * 70)

    for name, to, tc, th, tl, tv in test_candles:
        # Append candle de teste
        o = np.append(wo, to)
        c = np.append(wc, tc)
        h = np.append(wh, th)
        l = np.append(wl, tl)
        v = np.append(wv, tv)

        # Original (sem shadow boost)
        s_orig, _, _, _, _ = _candle_strength_core(
            o, c, h, l, v,
            window_size=100, short_window=20, volume_rel_min=1.0,
            shadow_alpha=2.0, shadow_weight=0.0,  # ← DESLIGADO
        )
        val_orig = s_orig[-1]

        # Enhanced (com shadow boost)
        s_enh, shape_e, size_e, vg_e, _ = _candle_strength_core(
            o, c, h, l, v,
            window_size=100, short_window=20, volume_rel_min=1.0,
            shadow_alpha=2.0, shadow_weight=0.40,  # ← LIGADO
        )
        val_enh = s_enh[-1]

        delta = val_enh - val_orig
        notes = ""
        if abs(delta) < 0.001:
            notes = "(sem mudança — não tem sombra favorável)"
        elif "Doji" in name and abs(val_orig) < 0.001:
            notes = f"(doji ativado! shape={shape_e[-1]:.3f})"
        elif abs(delta) > 0.05:
            notes = f"(shadow boost +{delta:+.3f})"

        print(f"  {name:<22} {val_orig:>+10.4f} {val_enh:>+10.4f} {delta:>+8.4f}  {notes}")

    # =================================================================
    # PARTE 2: DISTRIBUIÇÃO NATURAL DO ELEPHANT BAR
    # =================================================================
    print("-" * 76)
    print("PARTE 2: DISTRIBUIÇÃO — JÁ CONCENTRADA NATURALMENTE")
    print("-" * 76)

    # Gerar série longa para análise de distribuição
    np.random.seed(42)
    n = 5000
    price = 100.0
    o_all, c_all, h_all, l_all, v_all, ts_all = [], [], [], [], [], []
    base_dt = datetime(2024, 1, 2, tzinfo=timezone.utc)
    cpd = 78

    for i in range(n):
        day = i // cpd
        ci = i % cpd
        mins = ci * 5
        hr = 9 + (30 + mins) // 60
        mn = (30 + mins) % 60
        day_dt = base_dt + timedelta(days=day)
        ts_all.append(day_dt.replace(hour=hr, minute=mn))

        progress = ci / cpd
        vol_intra = 2.0 * (progress - 0.5) ** 2 + 0.3

        body = np.random.normal(0.02, vol_intra * 0.6)
        oi = price
        ci_v = oi + body
        hi = max(oi, ci_v) + abs(np.random.exponential(vol_intra * 0.3))
        li = min(oi, ci_v) - abs(np.random.exponential(vol_intra * 0.3))
        vi = max(100, 1000 * vol_intra + np.random.normal(0, 100))

        o_all.append(oi); c_all.append(ci_v)
        h_all.append(hi); l_all.append(li); v_all.append(vi)
        price = ci_v

    o_a = np.array(o_all); c_a = np.array(c_all)
    h_a = np.array(h_all); l_a = np.array(l_all)
    v_a = np.array(v_all); ts_a = np.array(ts_all)

    # Default (sem concentração extra)
    res_default = candle_strength_indicator(
        o_a, c_a, h_a, l_a, v_a,
        return_components=True)

    # Com concentração (para referência)
    res_conc = candle_strength_indicator(
        o_a, c_a, h_a, l_a, v_a,
        use_concentration=True, return_components=True)

    # Com ergódico
    res_full = candle_strength_indicator(
        o_a, c_a, h_a, l_a, v_a, timestamps=ts_a,
        return_components=True)

    print(f"""
  A estrutura multiplicativa + tanh do elephant bar já produz uma 
  distribuição naturalmente concentrada (diferente do quantile rank 
  do CSI puro, que é uniforme). Concentração extra é OPCIONAL.
    """)

    conc_p_val = res_conc['concentration_p']
    print(f"  {'Threshold':>12} {'Default':>10} {'+ Ergódico':>11} {f'+ Conc(p={conc_p_val:.1f})':>14}")
    print("  " + "-" * 50)

    for t in [0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9]:
        pd_ = np.mean(np.abs(res_default['csi']) > t) * 100
        pf = np.mean(np.abs(res_full['csi']) > t) * 100
        pc = np.mean(np.abs(res_conc['csi']) > t) * 100
        print(f"  |CSI|>{t:.2f}:  {pd_:>9.2f}% {pf:>10.2f}% {pc:>13.2f}%")

    print("""
  Default (sem concentração): distribuição já é conservadora.
  Valores > 0.8 são naturalmente raros (~1-3%).
  
  Concentração extra: disponível via use_concentration=True se necessário,
  mas com a estrutura do elephant bar normalmente NÃO é preciso.
  
  Gate ergódico: atenua candles típicos para seu horário, reduz falsos positivos.
    """)

    # =================================================================
    # PARTE 3: GATE ERGÓDICO
    # =================================================================
    print("-" * 76)
    print("PARTE 3: GATE ERGÓDICO")
    print("-" * 76)

    tf = res_full['timeframe_info']
    print(f"\n  Timeframe: {tf['label']} ({tf['category']})")

    tail = min(20 * cpd, n)
    start = n - tail
    hours = np.array([ts_all[i].hour + ts_all[i].minute / 60 for i in range(n)])
    periods = [
        ("Abertura (9:30-10:30)",  9.5, 10.5),
        ("Almoço  (12:00-13:00)", 12.0, 13.0),
        ("Fech.   (15:00-16:00)", 15.0, 16.0),
    ]

    print(f"\n  {'Período':<28} {'|CSI| sem':>10} {'|CSI| +erg':>11} "
          f"{'Gate méd':>9} {'Conf':>6}")
    print("  " + "-" * 66)

    for pname, t0, t1 in periods:
        mask = (hours[start:] >= t0) & (hours[start:] < t1)
        idx = np.where(mask)[0] + start
        if len(idx) == 0:
            continue
        csn = np.mean(np.abs(res_default['csi'][idx]))   # sem ergódico
        csf = np.mean(np.abs(res_full['csi'][idx]))       # com ergódico
        gm = np.mean(res_full['ergodic_gate'][idx])
        co = np.mean(res_full['ergodic_confidence'][idx])
        print(f"  {pname:<28} {csn:>10.4f} {csf:>11.4f} {gm:>9.4f} {co:>6.2f}")

    print("""
  Gate < 1.0: candles típicos para seu horário são atenuados.
  |CSI| com gate ≤ sem gate: NUNCA infla.
    """)

    # =================================================================
    # PARTE 4: DEGRADAÇÃO GRACEFUL
    # =================================================================
    print("-" * 76)
    print("PARTE 4: DEGRADAÇÃO GRACEFUL")
    print("-" * 76 + "\n")

    for nc in [10, 50, 200, 1000, n]:
        r = candle_strength_indicator(
            o_a[:nc], c_a[:nc], h_a[:nc], l_a[:nc], v_a[:nc],
            timestamps=ts_a[:nc], return_components=True)
        ac = np.mean(r['ergodic_confidence'])
        ag = np.mean(r['ergodic_gate'])
        pa = np.mean(r['ergodic_confidence'] > 0) * 100
        print(f"  {nc:>6} candles → Conf: {ac:.3f}  "
              f"Gate: {ag:.4f}  Buckets: {pa:.0f}%")

    print("""
  Poucos dados → gate=1.0 → elephant bar puro (original)
  Dados abundantes → gate ativo → proteção extra
    """)

    print("=" * 76)
    print("FIM — CSI v3 Hybrid")
    print("=" * 76)


if __name__ == "__main__":
    run_examples()

CSI v3 HYBRID — VALIDAÇÃO

----------------------------------------------------------------------------
PARTE 1: SHADOW BOOST — ORIGINAL vs ENHANCED
----------------------------------------------------------------------------

  Candle                   Original   Enhanced        Δ  Notas
  ----------------------------------------------------------------------
  Marubozu Bullish          +0.9787    +0.9787  +0.0000  (sem mudança — não tem sombra favorável)
  Marubozu Bearish          -0.9787    -0.9787  +0.0000  (sem mudança — não tem sombra favorável)
  Hammer clássico           +0.1885    +0.2768  +0.0883  (shadow boost ++0.088)
  Shooting Star             -0.1808    -0.2691  -0.0883  (shadow boost +-0.088)
  Dragonfly Doji            +0.6262    +0.7515  +0.1253  (shadow boost ++0.125)
  Gravestone Doji           -0.6262    -0.7515  -0.1253  (shadow boost +-0.125)
  Doji simétrico            +0.0000    +0.0000  +0.0000  (sem mudança — não tem sombra favorável)
  Hammer vol baixo     

#### Transformação dos dados

In [3]:
cs = candle_strength_indicator(
        df.open.values, df.high.values, df.low.values, df.close.values, df.volume.values, df.index.values, return_components=True,
        short_window=1000
    )
cs
    

{'csi': array([0.15059221, 0.        , 0.        , ..., 0.18613454, 0.18522374,
        0.17231265]),
 'strength_raw': array([0.15059221, 0.        , 0.        , ..., 0.25025369, 0.24760586,
        0.21546349]),
 'shape': array([0.4, 1. , 0. , ..., 1. , 1. , 1. ]),
 'size': array([0.18086326, 1.        , 1.        , ..., 0.18672453, 0.07732359,
        0.11095525]),
 'volume_gate': array([0.6       , 0.        , 0.        , ..., 0.55988587, 0.66457091,
        0.54485706]),
 'volume_rel': array([1.        , 0.34842065, 0.3118492 , ..., 0.97585034, 1.03908444,
        0.96670222]),
 'ergodic_gate': array([1.        , 1.        , 1.        , ..., 0.74378341, 0.74805882,
        0.79973017]),
 'ergodic_rank': array([0.5       , 0.5       , 0.5       , ..., 0.40360767, 0.39808379,
        0.32700977]),
 'ergodic_confidence': array([0., 0., 0., ..., 1., 1., 1.]),
 'concentration_p': None,
 'timeframe_info': {'median_seconds': 120.0,
  'label': '2min',
  'category': 'minute'}}

#### Visualização gráfica

In [4]:
df['csi'] = cs['csi']

In [2]:
import numpy as np
from numba import njit

@njit(nogil=True, cache=True)
def _bisect_left(a, n, x):
    lo = 0
    hi = n
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


@njit(nogil=True, cache=True)
def _quantile_from_sorted_prefix(sorted_vals, n, q):
    if n == 0:
        return np.nan

    if q <= 0.0:
        return sorted_vals[0]
    if q >= 1.0:
        return sorted_vals[n - 1]

    idx = q * (n - 1)
    lo = int(np.floor(idx))
    hi = int(np.ceil(idx))

    if lo == hi:
        return sorted_vals[lo]

    frac = idx - lo
    return sorted_vals[lo] + (sorted_vals[hi] - sorted_vals[lo]) * frac


@njit(nogil=True, cache=True)
def rolling_quantile_incremental(arr, window, q):
    """
    Rolling quantile exato, causal e incremental.
    
    Regras:
    - out[i] usa somente arr[max(0, i-window+1):i+1]
    - sem leitura para frente
    - se houver NaN na janela completa, out[i] = NaN
    - mesmo comportamento do seu código original para janela incompleta:
      índices < window-1 ficam como NaN
    """
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan

    if window <= 0:
        return out

    # Janela ordenada contendo apenas os valores válidos da janela atual
    sorted_vals = np.empty(window, dtype=np.float64)

    # Ring buffer com os valores brutos da janela para saber quem sai
    ring = np.empty(window, dtype=np.float64)
    ring[:] = np.nan

    # Quantidade de valores válidos atualmente na janela
    m = 0

    for i in range(n):
        slot = i % window

        # 1) Remove o valor que está saindo da janela
        if i >= window:
            old_v = ring[slot]

            if not np.isnan(old_v):
                pos = _bisect_left(sorted_vals, m, old_v)

                # Remove a primeira ocorrência encontrada
                # shift à esquerda
                for j in range(pos, m - 1):
                    sorted_vals[j] = sorted_vals[j + 1]

                m -= 1

        # 2) Insere o valor novo
        new_v = arr[i]
        ring[slot] = new_v

        if not np.isnan(new_v):
            pos = _bisect_left(sorted_vals, m, new_v)

            # shift à direita para abrir espaço
            for j in range(m, pos, -1):
                sorted_vals[j] = sorted_vals[j - 1]

            sorted_vals[pos] = new_v
            m += 1

        # 3) Só calcula quando a janela está completa
        #    e sem NaN (m == window)
        if i >= window - 1 and m == window:
            out[i] = _quantile_from_sorted_prefix(sorted_vals, m, q)

    return out

In [ ]:

import numpy as np
from numba import njit

@njit(nogil=True, cache=True)
def _bisect_left(a, n, x):
    lo = 0
    hi = n
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


@njit(nogil=True, cache=True)
def _quantile_from_sorted_prefix(sorted_vals, n, q):
    if n == 0:
        return np.nan

    if q <= 0.0:
        return sorted_vals[0]
    if q >= 1.0:
        return sorted_vals[n - 1]

    idx = q * (n - 1)
    lo = int(np.floor(idx))
    hi = int(np.ceil(idx))

    if lo == hi:
        return sorted_vals[lo]

    frac = idx - lo
    return sorted_vals[lo] + (sorted_vals[hi] - sorted_vals[lo]) * frac


@njit(nogil=True, cache=True)
def rolling_quantile_incremental(arr, window, q):
    """
    Rolling quantile exato, causal e incremental.
    
    Regras:
    - out[i] usa somente arr[max(0, i-window+1):i+1]
    - sem leitura para frente
    - se houver NaN na janela completa, out[i] = NaN
    - mesmo comportamento do seu código original para janela incompleta:
      índices < window-1 ficam como NaN
    """
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan

    if window <= 0:
        return out

    # Janela ordenada contendo apenas os valores válidos da janela atual
    sorted_vals = np.empty(window, dtype=np.float64)

    # Ring buffer com os valores brutos da janela para saber quem sai
    ring = np.empty(window, dtype=np.float64)
    ring[:] = np.nan

    # Quantidade de valores válidos atualmente na janela
    m = 0

    for i in range(n):
        slot = i % window

        # 1) Remove o valor que está saindo da janela
        if i >= window:
            old_v = ring[slot]

            if not np.isnan(old_v):
                pos = _bisect_left(sorted_vals, m, old_v)

                # Remove a primeira ocorrência encontrada
                # shift à esquerda
                for j in range(pos, m - 1):
                    sorted_vals[j] = sorted_vals[j + 1]

                m -= 1

        # 2) Insere o valor novo
        new_v = arr[i]
        ring[slot] = new_v

        if not np.isnan(new_v):
            pos = _bisect_left(sorted_vals, m, new_v)

            # shift à direita para abrir espaço
            for j in range(m, pos, -1):
                sorted_vals[j] = sorted_vals[j - 1]

            sorted_vals[pos] = new_v
            m += 1

        # 3) Só calcula quando a janela está completa
        #    e sem NaN (m == window)
        if i >= window - 1 and m == window:
            out[i] = _quantile_from_sorted_prefix(sorted_vals, m, q)

    return out

def wick_detection(data):
    
    high = np.ascontiguousarray(data["high"].to_numpy(dtype=np.float64, copy=False))
    low = np.ascontiguousarray(data["low"].to_numpy(dtype=np.float64, copy=False))
    close = np.ascontiguousarray(data["close"].to_numpy(dtype=np.float64, copy=False))
    open_ = np.ascontiguousarray(data["open"].to_numpy(dtype=np.float64, copy=False))    
    ts_ns = data.index.asi8
    n = len(close)

    
    # --- Wick absolutos e ratios ---
    bar_range = high - low
    body_bottom = np.minimum(open_, close)
    body_top = np.maximum(open_, close)
    lower_wick_abs = body_bottom - low          # em pontos
    upper_wick_abs = high - body_top            # em pontos

    # Ratios (proporção do range)
    safe_range = np.where(bar_range > 1e-12, bar_range, 1e-12)
    lower_wick_ratio = lower_wick_abs / safe_range
    upper_wick_ratio = upper_wick_abs / safe_range

    # --- Rolling quantile 80 (janela = 100, hardcoded) ---
    q_window = 5000
    q_level = 0.995

    lw_abs_q80 = rolling_quantile_incremental(lower_wick_abs, q_window, q_level)
    uw_abs_q80 = rolling_quantile_incremental(upper_wick_abs, q_window, q_level)
    lw_ratio_q80 = rolling_quantile_incremental(lower_wick_ratio, q_window, q_level)
    uw_ratio_q80 = rolling_quantile_incremental(upper_wick_ratio, q_window, q_level)

    df['lower_wick_detection'] = lw_abs_q80
    return df

df = wick_detection(df)
df

,open,high,low,close,volume,lower_wick_detection
2017-01-03 07:00:00+00:00,4895.7,4895.7,4895.4,4895.6,0.00338,NaN
2017-01-03 07:02:00+00:00,4895.4,4896.4,4895.4,4896.2,0.00117,NaN
2017-01-03 07:04:00+00:00,4896.1,4896.2,4896.0,4896.2,0.00104,NaN
2017-01-03 07:06:00+00:00,4896.1,4897.2,4896.0,4896.1,0.00273,NaN
2017-01-03 07:08:00+00:00,4896.2,4896.5,4896.1,4896.1,0.00065,NaN
...,...,...,...,...,...,...
2025-12-19 16:55:00+00:00,25284.8,25290.0,25271.6,25272.6,0.03770,17.5005
2025-12-19 16:56:00+00:00,25272.6,25277.2,25266.1,25273.0,0.03500,17.5005
2025-12-19 16:57:00+00:00,25273.0,25276.9,25268.2,25270.2,0.03220,17.5005
2025-12-19 16:58:00+00:00,25271.9,25273.5,25263.6,25265.9,0.03430,17.5005


In [9]:
df_plot = df.copy()
if 'time' not in df_plot.columns:
    df_plot = df_plot.rename_axis('time').reset_index()

df_plot['time'] = pd.to_datetime(df_plot['time'], utc=True, errors='coerce').dt.tz_convert(None)
df_plot = df_plot.dropna(subset=['time']).sort_values('time').drop_duplicates('time', keep='last')



In [13]:
body_bottom = np.minimum(df.open.values, df.close.values)
lower_wick_abs = body_bottom - df.low.values
wick_detected = lower_wick_abs >= df_plot.lower_wick_detection
df_plot['wick_detected'] = wick_detected


In [14]:
from lightweight_charts import Chart, JupyterChart
chart = Chart(height=750, width=1000)
chart.set(    
    df_plot,
    engine='duckdb',
    engine_options={
        'initial_bars': 3000,
        'chunk_bars': 1500,
        'prefetch_bars': 300,
        'max_bars': 20000,
    },
    
    indicators={
        'wick_detected': {'pane': 'osc', 'type': 'histogram'}
        
    }
          
)
chart.show(block=True)

/Users/marcelogarcia/Documents/Development/lightweight-charts-python/lightweight_charts/abstract.py:785: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  indicator_df.loc[~valid_ohlc_mask, indicator] = pd.NA
/Users/marcelogarcia/Documents/Development/lightweight-charts-python/lightweight_charts/chart.py:213: RuntimeWarning: Chart.show(block=True) was called inside an active event loop. Starting chart listener as a background task; use await chart.show_async() for notebook-friendly blocking behavior.
  warnings.warn(


<Task pending name='Task-7' coro=<Chart.show_async() running at /Users/marcelogarcia/Documents/Development/lightweight-charts-python/lightweight_charts/chart.py:222>>

/Users/marcelogarcia/Documents/Development/lightweight-charts-python/lightweight_charts/abstract.py:785: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  indicator_df.loc[~valid_ohlc_mask, indicator] = pd.NA


#### Outros

In [ ]:
import lightweight_charts, inspect
from lightweight_charts import Chart
print(lightweight_charts.__file__)
print(inspect.signature(Chart.set))

In [8]:
#!pip install -e /Users/marcelogarcia/Documents/Development/lightweight-charts-python
!pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 8.3 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
df[['open','high','low','close']].isna().any(axis=1).sum()
